# Benchmark: ChebAr vs plain Arnoldi vs naive Arnoldi

**Cat size.** `alpha_sq` is `|alpha|^2` (`epsilon_d = 2 alpha_sq g`, `docs/models_ats.md` §2). Saved outputs predate the 2026-09 fix and print twice the current values for the same physical point.

Three ways to get the same number — the bit-flip rate, i.e. the decay rate of
the slowest non-trivial eigenvalue of the trace-projected Floquet propagator —
compared at `alpha_sq = 1.5` and `alpha_sq = 3`, on identical Hilbert spaces and
on the same GPU.

| key | solver | block | what it is |
|---|---|---|---|
| `cheb_ar` | `ChebAr` | `n_periods = 6` | Chebyshev-filtered Arnoldi, `notebooks/solvers/cheb_ar.ipynb` |
| `arnoldi` | `ArnoldiLindblad` | `n_periods = 6` | unfiltered Arnoldi, `notebooks/solvers/arnoldi_no_cheb.ipynb` |
| `arnoldi_naiv` | `ArnoldiLindblad` | `n_periods = 1` | unfiltered Arnoldi on a single drive period, `notebooks/solvers/arnoldi_no_cheb_naiv.ipynb` |

## What is being compared, and the rescaling

Each method produces a `mu_list`: the tracked Ritz value after every Arnoldi
step. Those `mu` are **not** on a common scale — they are eigenvalues of
different operators:

- `cheb_ar` and `arnoldi` propagate over `T_block = 6 * T_drive`, so
  `mu ~ exp(-rate * 6 * T_drive)`;
- `arnoldi_naiv` propagates over `T_block = T_drive`, so its `mu` is the sixth
  root of the other two (`mu_naiv**6 == mu_6periods`, up to integrator noise).

Rather than rescale `mu` by hand, everything is plotted as the **predicted
bit-flip rate**, `rate_k = -log(mu_k) / T_block`, which is the same physical
quantity in the same units for all three — `T_block` absorbs the rescaling
exactly.

## The cost axis

Arnoldi steps are not comparable across methods either: one `ChebAr` step
applies the propagator `cheb_degree` times (plus a one-off `m_arnoldi_0`-step
unfiltered estimation before the filter can be built), while one plain Arnoldi
step applies it once. So the convergence plots run against **propagator
applications**:

```
apps(k) = k                             # arnoldi, arnoldi_naiv
apps(k) = m_arnoldi_0 + cheb_degree * k # cheb_ar
```

One application is still not the same amount of work everywhere — a
`arnoldi_naiv` application integrates one drive period, the others six — so the
same curves are also plotted against **estimated wall clock**,
`t_apply * apps(k)`, with `t_apply` measured on the worker (one timed
propagator call after warm-up, outside any Arnoldi loop). That axis is the one
to read for "which method is actually faster".

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
2. Run all cells.

Two cluster jobs, one per `alpha_sq`, each running all three methods
back-to-back on a single GPU — so the timings within a job are directly
comparable (same device, one model build per frame, no cross-job variance).
Expect roughly **10–20 minutes per job**; they are submitted sequentially.

See `notebooks/solvers/cheb_ar.ipynb` for the auth/bootstrap details (the cell below is
copied from it verbatim) and for the Fock-truncation discussion behind
`n_a`/`n_b`.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

In [ ]:
def run_benchmark(
    alpha_sq=3.0,
    n_a=25,
    n_b=11,
    epsilon_p=0.1,
    n_periods=6,
    # ChebAr
    m_cheb_0=60,
    m_cheb=90,
    cheb_degree=6,
    margin=1e-2,
    # plain Arnoldi, 6-period block
    m_arnoldi=90,
    # naive Arnoldi, 1-period block
    m_naiv=90*6,
    seed=0,
):
    """Run all three methods at one `alpha_sq`, on one GPU worker.

    One job per parameter point rather than one per (method, point): the three
    methods then share a device and a process, which is what makes their
    timings comparable at all.

    Returns plain numpy/python objects only -- the client has neither jax nor
    dynamiqs. In particular the Krylov basis `Q` (hundreds of MB) never crosses
    the wire; only `mu_list`, the step/cost bookkeeping and the timings do.
    """
    import time
    import traceback

    import numpy as np

    import dynamiqs as dq
    from floquet_lindblad import ArnoldiLindblad, ChebAr
    from floquet_lindblad.models.ats import build_ats_hamiltonian_interaction

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    def build(np_periods):
        """Interaction-frame model for a block of `np_periods` drive periods."""
        return build_ats_hamiltonian_interaction(
            n_a=n_a,
            n_b=n_b,
            alpha_sq=alpha_sq,
            epsilon_p=epsilon_p,
            n_periods=np_periods,
        )

    def time_apply(fn, x, repeats=3):
        """Seconds per application of `fn`, jit warm-up excluded.

        `block_until_ready` is mandatory: jax dispatch is asynchronous, so
        without it this measures the launch, not the solve.
        """
        fn(x).block_until_ready()  # warm-up: trace + compile
        t = time.time()
        for _ in range(repeats):
            y = fn(x)
        y.block_until_ready()
        return (time.time() - t) / repeats

    out = {}
    params = None  # set by the first build() that gets that far

    # ------------------------------------------------------------------ #
    # 1. ChebAr, 6-period block                                           #
    # ------------------------------------------------------------------ #
    try:
        t0 = time.time()
        H_I, jops, jops_LdL, phase, V_cheb, T_cheb, params = build(1)
        solver = ChebAr(
            H_I,
            jops,
            T_cheb,
            jump_ops_LdL=jops_LdL,
            dims=(n_a, n_b),
            output_phase=phase,
            cheb_degree=cheb_degree,
        )
        x0 = solver.make_x0(seed=seed)
        t_build = time.time() - t0

        t0 = time.time()
        _, _, ritz_vals = solver.first_estimation(x0, m_arnoldi=m_cheb_0)
        t_est = time.time() - t0

        solver.setup_chebyshev(ritz_vals, margin=margin)

        t0 = time.time()
        Q, H, mu_list = solver.arnoldi_hessenberg(
            x0, solver.chebyshev_filter, m_cheb, warm_start=False
        )
        t_run = time.time() - t0

        x_ritz, _ = solver.ritz_vector(Q, H, m_cheb)
        res = solver.residual_check(x_ritz)
        del Q, x_ritz  # the Krylov basis must not survive into the next method

        # mu_list[j] comes from the (j+1)x(j+1) Hessenberg block, i.e. step j+1.
        steps = np.arange(1, len(mu_list) + 1)
        out["cheb_ar"] = {
            "label": "ChebAr (filtered, 6 periods)",
            "n_periods": n_periods,
            "T_block": py(T_cheb),
            "steps": steps,
            # one filtered application = `cheb_degree` propagator applications,
            # on top of the one-off unfiltered first estimation
            "apps": m_cheb_0 + cheb_degree * steps,
            "mu_list": np.asarray(mu_list),
            "t_apply": time_apply(solver.propagate_block_projected, x0),
            "t_apply_filter": time_apply(solver.chebyshev_filter, x0),
            "t_build": t_build,
            "t_setup": t_est,
            "t_run": t_run,
            "m": m_cheb,
            "mu_RQ": py(res["mu_RQ"]),
            "rate_RQ": py(res["rate_RQ"]),
            "res_rel": py(res["res_rel"]),
            "ellipse": {k: py(v) for k, v in solver.ellipse.items() if np.ndim(v) == 0},
        }
        del solver
    except Exception:
        out["cheb_ar"] = {"error": traceback.format_exc()}

    # ------------------------------------------------------------------ #
    # 2 & 3. Unfiltered Arnoldi: 6-period block, then 1-period ("naive")  #
    # ------------------------------------------------------------------ #
    for key, np_periods, m_req, label in (
        ("arnoldi", n_periods, m_arnoldi, "Arnoldi (unfiltered, 6 periods)"),
        ("arnoldi_naiv", 1, m_naiv, "Arnoldi naive (unfiltered, 1 period)"),
    ):
        try:
            t0 = time.time()
            H_I, jops, jops_LdL, phase, V, T_block, params = build(np_periods)
            solver = ArnoldiLindblad(
                H_I,
                jops,
                T_block,
                jump_ops_LdL=jops_LdL,
                dims=(n_a, n_b),
                output_phase=phase,
            )
            x0 = solver.make_x0(seed=seed)
            t_build = time.time() - t0

            t0 = time.time()
            Q, H, steps, mu_list = solver.arnoldi(x0, m_req, stride=1)
            t_run = time.time() - t0

            # `steps[-1]` is what the factorization reached and where
            # `mu_list[-1]` came from; a smaller m would discard Krylov vectors.
            m_reached = int(steps[-1])
            x_ritz, _ = solver.ritz_vector(Q, H, m_reached, target=mu_list[-1])
            res = solver.residual_check(x_ritz)
            H = np.asarray(H)
            del Q, x_ritz

            out[key] = {
                "label": label,
                "n_periods": np_periods,
                "T_block": py(T_block),
                "steps": np.asarray(steps),
                "apps": np.asarray(steps),  # one step = one propagator call
                "mu_list": np.asarray(mu_list),
                "t_apply": time_apply(solver.propagate, x0),
                "t_build": t_build,
                "t_setup": 0.0,
                "t_run": t_run,
                "m": m_reached,
                "mu_RQ": py(res["mu_RQ"]),
                "rate_RQ": py(res["rate_RQ"]),
                "res_rel": py(res["res_rel"]),
                # H is real up to integrator noise: a free noise-floor estimate
                "H_imag_max": float(np.abs(H.imag).max()),
            }
            del solver
        except Exception:
            out[key] = {"error": traceback.format_exc()}

    import jax

    return {
        "methods": out,
        "params": None if params is None else {k: py(v) for k, v in params.items()},
        "settings": {
            "alpha_sq": alpha_sq,
            "n_a": n_a,
            "n_b": n_b,
            "epsilon_p": epsilon_p,
            "n_periods": n_periods,
            "m_cheb_0": m_cheb_0,
            "m_cheb": m_cheb,
            "cheb_degree": cheb_degree,
            "margin": margin,
            "m_arnoldi": m_arnoldi,
            "m_naiv": m_naiv,
            "seed": seed,
        },
        "device": str(jax.devices()[0]),
    }

In [ ]:
import time

ALPHA_SQ = [1.5, 3.0]

# Sequential, not concurrent: the point of the exercise is the timings, and two
# jobs racing for the same pool would make them meaningless.
results = {}
for alpha_sq in ALPHA_SQ:
    t0 = time.time()
    results[alpha_sq] = acr.run(
        run_benchmark,
        alpha_sq=alpha_sq,
        num_gpus=1,
        num_cpus=8,
        runtime_env=RUNTIME_ENV,
        poll_seconds=5,
    )
    r = results[alpha_sq]
    print(f"\nalpha_sq = {alpha_sq}: {time.time() - t0:.1f} s wall (incl. startup)")
    print("  device:", r["device"])
    for key, m in r["methods"].items():
        if "error" in m:
            print(f"  {key:13s} FAILED")
        else:
            print(
                f"  {key:13s} build {m['t_build']:6.1f} s | "
                f"setup {m['t_setup']:6.1f} s | run {m['t_run']:7.1f} s | "
                f"t_apply {m['t_apply'] * 1e3:7.1f} ms"
            )

# Surface any failure in full rather than letting the plots go quietly empty.
for alpha_sq, r in results.items():
    for key, m in r["methods"].items():
        if "error" in m:
            print(f"\n=== alpha_sq={alpha_sq}, {key} ===\n{m['error']}")

In [ ]:
import numpy as np

# Fixed hue per method, never cycled; marker and dash carry the same identity so
# the curves stay separable in grayscale and for colorblind readers.
STYLE = {
    "cheb_ar": {"color": "#0072B2", "marker": "o", "ls": "-"},
    "arnoldi": {"color": "#E69F00", "marker": "s", "ls": "--"},
    "arnoldi_naiv": {"color": "#009E73", "marker": "^", "ls": ":"},
}
ORDER = ["cheb_ar", "arnoldi", "arnoldi_naiv"]


def rate_of(m):
    """Predicted bit-flip rate after each Arnoldi step.

    `-log(mu_k) / T_block` -- the one form in which the three methods are
    directly comparable, since T_block absorbs the 6-periods-vs-1-period
    rescaling of `mu` (mu_naiv**6 == mu_6periods).
    """
    return (-np.log(np.asarray(m["mu_list"], dtype=complex)) / m["T_block"]).real


def ok(r):
    """The methods of run `r` that did not raise, in fixed plotting order."""
    return [(k, r["methods"][k]) for k in ORDER if "error" not in r["methods"][k]]


def reference_rate(r):
    """Converged rate to measure against: the unfiltered 6-period Rayleigh quotient.

    That is the reference the repo already treats as ground truth (see
    docs/arnoldi_no_cheb.md); fall back to whichever method did run.
    """
    for key in ("arnoldi", "arnoldi_naiv", "cheb_ar"):
        m = r["methods"].get(key, {})
        if "error" not in m:
            return complex(m["rate_RQ"]).real, key
    raise RuntimeError("no method succeeded")


# --- converged answers, and whether the three agree ------------------------
for alpha_sq, r in results.items():
    ref, ref_key = reference_rate(r)
    print(f"\nalpha_sq = {alpha_sq}   (reference: {ref_key}, rate_RQ = {ref:.6e})")
    print(
        f"  {'method':13s} {'m':>4} {'apps':>6} {'rate_bf':>13} "
        f"{'rel. dev.':>10} {'res_rel':>10} {'run [s]':>9}"
    )
    for key, m in ok(r):
        rate = rate_of(m)[-1]
        print(
            f"  {key:13s} {m['m']:4d} {int(m['apps'][-1]):6d} {rate:13.6e} "
            f"{abs(rate - ref) / abs(ref):10.2e} {abs(m['res_rel']):10.2e} "
            f"{m['t_run']:9.1f}"
        )

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1, len(results), figsize=(5.2 * len(results), 4.0), constrained_layout=True
)
axes = np.atleast_1d(axes)

for ax, (alpha_sq, r) in zip(axes, results.items()):
    ref, _ = reference_rate(r)
    for key, m in ok(r):
        ax.plot(
            m["apps"],
            np.abs(rate_of(m)),
            **STYLE[key],
            ms=3.5,
            lw=1.2,
            markevery=max(1, len(m["apps"]) // 25),
            label=m["label"],
        )
    ax.axhline(abs(ref), color="0.35", lw=0.8, ls="-", zorder=0)
    ax.annotate(
        "converged rate",
        xy=(0.98, abs(ref)),
        xycoords=("axes fraction", "data"),
        ha="right",
        va="bottom",
        fontsize=7,
        color="0.35",
    )
    ax.set_yscale("log")
    ax.set_xlabel("propagator applications")
    ax.set_title(rf"$|\alpha|^2 = {alpha_sq}$")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)

axes[0].set_ylabel(r"predicted bit-flip rate $|{-}\log\mu_k / T_{\rm block}|$")
axes[0].legend(fontsize=7, frameon=False)
fig.suptitle("Predicted bit-flip rate vs cost", fontsize=10)
plt.show()

In [ ]:
# Same convergence, against estimated wall clock instead of applications.
#
# Applications are not equal work across methods: a 1-period application
# integrates a sixth of the time a 6-period one does, and a ChebAr step is
# `cheb_degree` applications. `t_apply` is measured per method on the worker
# (timed call after warm-up), so `t_apply * apps` is the honest cost axis --
# an estimate, not the measured Arnoldi wall clock, which also carries
# orthogonalization and the one-off setup.

fig, axes = plt.subplots(
    1, len(results), figsize=(5.2 * len(results), 4.0), constrained_layout=True
)
axes = np.atleast_1d(axes)

for ax, (alpha_sq, r) in zip(axes, results.items()):
    ref, ref_key = reference_rate(r)
    for key, m in ok(r):
        # ChebAr pays for its unfiltered first estimation before step 1, which
        # `apps` already includes.
        ax.plot(
            np.asarray(m["apps"]) * m["t_apply"],
            rate_of(m),
            **STYLE[key],
            ms=3.5,
            lw=1.2,
            markevery=max(1, len(m["apps"]) // 25),
            label=m["label"],
        )
    ax.axhline(abs(r["methods"][ref_key]["res_rel"]), color="0.35", lw=0.8, ls="--")
    ax.set_yscale("log")
    ax.set_xlabel("estimated GPU time [s]")
    ax.set_title(rf"$|\alpha|^2 = {alpha_sq}$")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)

axes[0].set_ylabel(r"predicted bit-flip rate $|{-}\log\mu_k / T_{\rm block}|$")
axes[0].legend(fontsize=7, frameon=False)
fig.suptitle("Convergence vs estimated wall clock", fontsize=10)
plt.show()

In [ ]:
# Measured timings, as reported by the worker.
#
# `run` is the Arnoldi loop itself; `setup` is ChebAr's unfiltered first
# estimation plus the ellipse fit (zero for the others); `build` is the model +
# solver construction, which includes jax tracing/compilation and is therefore
# noisy -- it is shown for completeness, not as a property of the method.

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)

rows = [
    (f"{key}\n" + rf"$|\alpha|^2={alpha_sq:g}$", key, m)
    for alpha_sq, r in results.items()
    for key, m in ok(r)
]
labels = [lab for lab, _, _ in rows]
colors = [STYLE[key]["color"] for _, key, _ in rows]
run = np.array([m["t_run"] for _, _, m in rows])
setup = np.array([m["t_setup"] for _, _, m in rows])
build = np.array([m["t_build"] for _, _, m in rows])
t_apply_ms = np.array([m["t_apply"] * 1e3 for _, _, m in rows])

y = np.arange(len(rows))
ax = axes[0]
ax.barh(y, run, color=colors, height=0.55, label="Arnoldi loop")
ax.barh(y, setup, left=run, color="0.55", height=0.55, label="ChebAr setup")
ax.barh(y, build, left=run + setup, color="0.82", height=0.55,
        label="build + compile")
for yi, total in zip(y, run + setup + build):
    ax.text(total, yi, f" {total:.0f} s", va="center", fontsize=7, color="0.25")
ax.set_yticks(y, labels, fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("seconds")
ax.set_title("Measured time to the reported answer", fontsize=9)
ax.legend(fontsize=7, frameon=False)

# Per-application cost: what the wall-clock axis above is built from.
ax = axes[1]
ax.barh(y, t_apply_ms, color=colors, height=0.55)
for yi, v in zip(y, t_apply_ms):
    ax.text(v, yi, f" {v:.0f} ms", va="center", fontsize=7, color="0.25")
ax.set_yticks(y, labels, fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("milliseconds per propagator application")
ax.set_title("Cost of one propagator application", fontsize=9)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, axis="x", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

## How to read this

- **Rate panels** — all three curves must land on the same horizontal line. If
  they do not, the disagreement is the finding: the filtered answer is the
  suspect (`margin`, `cheb_degree`, the ellipse fit), not the physics — see
  `docs/arnoldi_no_cheb.md`.
- **Convergence panels** — the tail of each curve is bounded by that method's
  `res_rel` and, below that, by the integrator noise floor (`H_imag_max` for the
  unfiltered runs). A curve flattening at `1e-9` has converged, not stalled.
- **Wall-clock panel** — this is the one that decides which method to use. The
  application-count panel systematically favours `arnoldi_naiv`, whose
  applications are six times cheaper but six times less informative.
- The `arnoldi` reference is only a reference up to its own residual; where all
  three agree to `1e-9`, the ranking between them is noise.